# 2D SH-Wave Site-Response Extension

**CEE 490 — Computer Methods in Structural/Geotechnical Engineering**

This notebook extends the 1D SH-wave velocity-stress finite-difference model
from `01_run_project.ipynb` into a full **2D SH-wave** simulation in a vertical
cross-section (x–z plane).  Particle motion remains out-of-plane ($v_y$),
but two stress components ($\\tau_{xy}$, $\\tau_{zy}$) now govern lateral
spreading, diffraction, and basin focusing that the 1D model cannot capture.

## Governing equations

$$\\rho(x,z)\\frac{\\partial v_y}{\\partial t}
= \\frac{\\partial \\tau_{xy}}{\\partial x}
+ \\frac{\\partial \\tau_{zy}}{\\partial z}$$

$$\\frac{\\partial \\tau_{xy}}{\\partial t}
= \\mu(x,z)\\frac{\\partial v_y}{\\partial x}, \\qquad
\\frac{\\partial \\tau_{zy}}{\\partial t}
= \\mu(x,z)\\frac{\\partial v_y}{\\partial z}$$

## Staggered-grid layout

| Field | Shape | Location |
|---|---|---|
| $v_y$ | `(nz, nx)` | primary nodes |
| $\\tau_{xy}$ | `(nz, nx-1)` | x-staggered |
| $\\tau_{zy}$ | `(nz-1, nx)` | z-staggered (no row above surface) |

## Outline
1. Configuration and CFL check
2. Material models (homogeneous, layered, basin)
3. Homogeneous verification: wavefront shape and arrival time
4. Layered model: reflections and free-surface response
5. Surface receiver array
6. Basin model
7. 2D wavefield animations
8. Limitations and next steps

---
## 0. Imports and path setup

In [ ]:
import sys
from pathlib import Path

# Make the project src directory importable from the notebook
project_root = Path().resolve().parent
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from site_response.materials_2d import (
    Simulation2DConfig,
    homogeneous_2d_model,
    layered_2d_model,
    basin_2d_model,
    format_2d_parameter_summary,
    ensure_2d_output_directories,
    cfl_check_2d,
)
from site_response.fd_solver_2d import run_2d_solver
from site_response.analysis_2d import (
    run_arrival_verification_table,
    format_arrival_table,
    compute_surface_peak_amplitudes,
    compute_2d_amplification,
)
from site_response.visualization_2d import (
    plot_vs_map,
    plot_impedance_map,
    plot_source_wavelet_2d,
    plot_velocity_snapshot,
    plot_receiver_records,
    plot_surface_array,
    plot_arrival_check,
    make_2d_wave_animation,
)

print('Imports OK')

---
## 1. Configuration and CFL check

In [ ]:
cfg = Simulation2DConfig(
    model_width_m=800.0,
    model_depth_m=400.0,
    dx_m=4.0,
    dz_m=4.0,
    time_step_s=0.0015,
    total_time_s=2.0,
    dominant_frequency_hz=4.0,
    soil_layer_thickness_m=50.0,
    soil_density_kg_m3=1800.0,
    soil_shear_velocity_m_s=250.0,
    rock_density_kg_m3=2200.0,
    rock_shear_velocity_m_s=800.0,
    absorbing_layer_thickness_m=80.0,
    absorbing_strength=3.5,
)

ensure_2d_output_directories(cfg)
print(format_2d_parameter_summary(cfg))

---
## 2. Material models

In [ ]:
mat_homo = homogeneous_2d_model(cfg)
mat_layered = layered_2d_model(cfg)
mat_basin = basin_2d_model(cfg, base_depth_m=20.0, extra_depth_m=60.0, basin_half_width_m=180.0)

print(f'Homogeneous  Vs range: {mat_homo.shear_velocity_m_s.min():.0f}–{mat_homo.shear_velocity_m_s.max():.0f} m/s')
print(f'Layered      Vs range: {mat_layered.shear_velocity_m_s.min():.0f}–{mat_layered.shear_velocity_m_s.max():.0f} m/s')
print(f'Basin        Vs range: {mat_basin.shear_velocity_m_s.min():.0f}–{mat_basin.shear_velocity_m_s.max():.0f} m/s')

In [ ]:
fig = plot_vs_map(mat_homo, output_path=cfg.figures_2d_dir / 'vs_map_homogeneous.png',
                  title='Homogeneous rock — Vs map')
plt.show()
plt.close(fig)

In [ ]:
fig = plot_vs_map(mat_layered, output_path=cfg.figures_2d_dir / 'vs_map_layered.png',
                  title='Layered soil over rock — Vs map')
plt.show()
plt.close(fig)

In [ ]:
fig = plot_impedance_map(mat_layered, output_path=cfg.figures_2d_dir / 'impedance_map_layered.png')
plt.show()
plt.close(fig)

In [ ]:
fig = plot_vs_map(mat_basin, output_path=cfg.figures_2d_dir / 'vs_map_basin.png',
                  title='Gaussian basin — Vs map')
plt.show()
plt.close(fig)

---
## 3. Homogeneous model — wavefront verification

A point source at the centre of the domain at depth 200 m should produce
an approximately circular wavefront in a uniform medium.  We also check
that the numerical peak arrival at a known receiver matches the theoretical
value $t = t_0 + d / V_s$.

In [ ]:
# Receivers: surface array + one internal verification receiver
SRC_X = cfg.model_width_m / 2   # 400 m
SRC_Z = 200.0

receivers_homo = {
    'surf_100': (100.0, 0.0),
    'surf_200': (200.0, 0.0),
    'surf_300': (300.0, 0.0),
    'surf_400': (400.0, 0.0),
    'surf_500': (500.0, 0.0),
    'surf_600': (600.0, 0.0),
    'surf_700': (700.0, 0.0),
    'internal': (SRC_X + 100.0, SRC_Z),   # 100 m offset at same depth
}

print('Running homogeneous 2D simulation …')
result_homo = run_2d_solver(
    cfg, mat_homo,
    source_x_m=SRC_X, source_z_m=SRC_Z,
    receivers=receivers_homo,
    store_every=10,
    source_scale=1.0e-4,
)
print(f'Done.  {len(result_homo.stored_time_s)} snapshots stored.')

In [ ]:
fig = plot_source_wavelet_2d(result_homo, output_path=cfg.figures_2d_dir / 'source_wavelet_2d.png')
plt.show()
plt.close(fig)

In [ ]:
# Pick two snapshot indices at roughly t=0.4 s and t=0.8 s
snap_times = result_homo.stored_time_s
idx_t1 = int(np.argmin(np.abs(snap_times - 0.4)))
idx_t2 = int(np.argmin(np.abs(snap_times - 0.8)))

fig = plot_velocity_snapshot(
    result_homo, idx_t1,
    output_path=cfg.figures_2d_dir / 'homogeneous_snapshot_t1.png',
    overlay_circle=True,
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_velocity_snapshot(
    result_homo, idx_t2,
    output_path=cfg.figures_2d_dir / 'homogeneous_snapshot_t2.png',
    overlay_circle=True,
)
plt.show()
plt.close(fig)

### 3.1 Arrival-time verification table

In [ ]:
checks = run_arrival_verification_table(result_homo)
print(format_arrival_table(checks))

# Save as CSV
import csv
csv_path = cfg.results_2d_dir / 'homogeneous_2d_arrival_check.csv'
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'receiver_name', 'receiver_x_m', 'receiver_z_m', 'distance_m',
        'theoretical_peak_s', 'numerical_peak_s', 'absolute_error_s', 'percent_error'
    ])
    writer.writeheader()
    for c in checks:
        writer.writerow({
            'receiver_name': c.receiver_name,
            'receiver_x_m': round(c.receiver_x_m, 2),
            'receiver_z_m': round(c.receiver_z_m, 2),
            'distance_m': round(c.distance_m, 2),
            'theoretical_peak_s': round(c.theoretical_peak_s, 6),
            'numerical_peak_s': round(c.numerical_peak_s, 6),
            'absolute_error_s': round(c.absolute_error_s, 6),
            'percent_error': round(c.percent_error, 3),
        })
print(f'Saved → {csv_path}')

In [ ]:
# Detailed plot for the internal receiver
c_int = next(c for c in checks if c.receiver_name == 'internal')
fig = plot_arrival_check(
    result_homo, 'internal',
    c_int.theoretical_peak_s, c_int.numerical_peak_s,
    output_path=cfg.figures_2d_dir / 'homogeneous_arrival_check.png',
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_receiver_records(
    result_homo,
    output_path=cfg.figures_2d_dir / 'homogeneous_receiver_records.png',
)
plt.show()
plt.close(fig)

In [ ]:
# Save homogeneous results
np.savez_compressed(
    cfg.results_2d_dir / 'homogeneous_2d_results.npz',
    time_s=result_homo.time_s,
    x_m=result_homo.x_m,
    z_m=result_homo.z_m,
    stored_time_s=result_homo.stored_time_s,
    velocity_snapshots_m_s=result_homo.velocity_snapshots_m_s,
    **{f'rec_{k}': v for k, v in result_homo.receiver_records_m_s.items()},
)
print('Homogeneous results saved.')

---
## 4. Layered model — reflections and free-surface response

A 50 m soft-soil layer (Vs = 250 m/s) over rock (Vs = 800 m/s).
Expected behavior: reflections at the interface, slower wavefronts in soil,
and reverberation at the free surface.

In [ ]:
# Use a deeper source (300 m) so the wave must travel through the interface
receivers_layered = {
    'surf_100': (100.0, 0.0),
    'surf_200': (200.0, 0.0),
    'surf_300': (300.0, 0.0),
    'surf_400': (400.0, 0.0),
    'surf_500': (500.0, 0.0),
    'surf_600': (600.0, 0.0),
    'surf_700': (700.0, 0.0),
    'in_soil':  (400.0, 25.0),    # inside soil layer
    'in_rock':  (400.0, 150.0),   # in rock below interface
}

print('Running layered 2D simulation …')
result_layered = run_2d_solver(
    cfg, mat_layered,
    source_x_m=400.0, source_z_m=300.0,
    receivers=receivers_layered,
    store_every=10,
    source_scale=1.0e-4,
)
print(f'Done.  {len(result_layered.stored_time_s)} snapshots stored.')

In [ ]:
snap_times_l = result_layered.stored_time_s
idx_lt1 = int(np.argmin(np.abs(snap_times_l - 0.5)))
idx_lt2 = int(np.argmin(np.abs(snap_times_l - 1.2)))

fig = plot_velocity_snapshot(
    result_layered, idx_lt1,
    output_path=cfg.figures_2d_dir / 'layered_snapshot_t1.png',
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_velocity_snapshot(
    result_layered, idx_lt2,
    output_path=cfg.figures_2d_dir / 'layered_snapshot_t2.png',
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_receiver_records(
    result_layered,
    output_path=cfg.figures_2d_dir / 'layered_receiver_records.png',
)
plt.show()
plt.close(fig)

In [ ]:
np.savez_compressed(
    cfg.results_2d_dir / 'layered_2d_results.npz',
    time_s=result_layered.time_s,
    x_m=result_layered.x_m,
    z_m=result_layered.z_m,
    stored_time_s=result_layered.stored_time_s,
    velocity_snapshots_m_s=result_layered.velocity_snapshots_m_s,
    **{f'rec_{k}': v for k, v in result_layered.receiver_records_m_s.items()},
)
print('Layered results saved.')

---
## 5. Surface receiver array

A wiggle-trace gather of all surface receivers reveals lateral variation in
arrival time and amplitude across the model.

In [ ]:
fig = plot_surface_array(
    result_layered,
    output_path=cfg.figures_2d_dir / 'layered_surface_array.png',
)
plt.show()
plt.close(fig)

In [ ]:
# Peak amplitudes along the surface for homogeneous vs layered
# (only surface receivers are at z=0)
surf_names_homo = [n for n, (rx, rz) in receivers_homo.items() if rz == 0.0]
surf_names_lay  = [n for n, (rx, rz) in receivers_layered.items() if rz == 0.0]

peaks_homo = compute_surface_peak_amplitudes(result_homo)
peaks_lay  = compute_surface_peak_amplitudes(result_layered)

xs_h = [receivers_homo[n][0] for n in surf_names_homo]
xs_l = [receivers_layered[n][0] for n in surf_names_lay]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xs_h, [peaks_homo[n] for n in surf_names_homo], 'o--', label='Homogeneous (deeper src)', color='steelblue')
ax.plot(xs_l, [peaks_lay[n]  for n in surf_names_lay],  's-',  label='Layered (deeper src)',     color='darkorange')
ax.set_xlabel('Receiver x (m)')
ax.set_ylabel('Peak |vy| (m/s)')
ax.set_title('Surface peak velocity — homogeneous vs layered')
ax.legend()
fig.tight_layout()
fig.savefig(cfg.figures_2d_dir / 'surface_peak_comparison.png', bbox_inches='tight')
plt.show()
plt.close(fig)

# Save summary CSV
import csv
with open(cfg.results_2d_dir / 'surface_receiver_summary.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['receiver', 'x_m', 'peak_homo_m_s', 'peak_layered_m_s'])
    for n in surf_names_lay:
        rx = receivers_layered[n][0]
        ph = peaks_homo.get(n, float('nan'))
        pl = peaks_lay[n]
        writer.writerow([n, rx, round(ph, 8), round(pl, 8)])
print('Surface summary saved.')

---
## 6. Basin model

A Gaussian-shaped soft basin demonstrates why 2D modeling matters:
edge-generated surface waves focus energy toward the basin center.

In [ ]:
receivers_basin = {
    'surf_100': (100.0, 0.0),
    'surf_200': (200.0, 0.0),
    'surf_300': (300.0, 0.0),
    'surf_400': (400.0, 0.0),   # basin center
    'surf_500': (500.0, 0.0),
    'surf_600': (600.0, 0.0),
    'surf_700': (700.0, 0.0),
    'in_rock':  (400.0, 200.0),
}

print('Running basin 2D simulation …')
result_basin = run_2d_solver(
    cfg, mat_basin,
    source_x_m=400.0, source_z_m=200.0,
    receivers=receivers_basin,
    store_every=10,
    source_scale=1.0e-4,
)
print(f'Done.  {len(result_basin.stored_time_s)} snapshots stored.')

In [ ]:
snap_times_b = result_basin.stored_time_s
idx_bt1 = int(np.argmin(np.abs(snap_times_b - 0.4)))
idx_bt2 = int(np.argmin(np.abs(snap_times_b - 1.0)))

fig = plot_velocity_snapshot(
    result_basin, idx_bt1,
    output_path=cfg.figures_2d_dir / 'basin_snapshot_t1.png',
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_velocity_snapshot(
    result_basin, idx_bt2,
    output_path=cfg.figures_2d_dir / 'basin_snapshot_t2.png',
)
plt.show()
plt.close(fig)

In [ ]:
fig = plot_surface_array(
    result_basin,
    output_path=cfg.figures_2d_dir / 'basin_surface_array.png',
)
plt.show()
plt.close(fig)

In [ ]:
np.savez_compressed(
    cfg.results_2d_dir / 'basin_2d_results.npz',
    time_s=result_basin.time_s,
    x_m=result_basin.x_m,
    z_m=result_basin.z_m,
    stored_time_s=result_basin.stored_time_s,
    velocity_snapshots_m_s=result_basin.velocity_snapshots_m_s,
    **{f'rec_{k}': v for k, v in result_basin.receiver_records_m_s.items()},
)
print('Basin results saved.')

---
## 7. Wavefield animations

Each animation is saved as a GIF to `figures_2d/`.

In [ ]:
from IPython.display import Image, display

# --- Homogeneous animation ---
anim_path_homo = cfg.figures_2d_dir / 'wavefield_2d_homogeneous.gif'
print(f'Saving homogeneous animation → {anim_path_homo}')
ani_h = make_2d_wave_animation(result_homo, output_path=anim_path_homo, fps=12, dpi=100)
plt.close('all')
display(Image(filename=str(anim_path_homo)))
print('Done.')

In [ ]:
# --- Layered animation ---
anim_path_lay = cfg.figures_2d_dir / 'wavefield_2d_layered.gif'
print(f'Saving layered animation → {anim_path_lay}')
ani_l = make_2d_wave_animation(result_layered, output_path=anim_path_lay, fps=12, dpi=100)
plt.close('all')
display(Image(filename=str(anim_path_lay)))
print('Done.')

In [ ]:
# --- Basin animation ---
anim_path_basin = cfg.figures_2d_dir / 'wavefield_2d_basin.gif'
print(f'Saving basin animation → {anim_path_basin}')
ani_b = make_2d_wave_animation(result_basin, output_path=anim_path_basin, fps=12, dpi=100)
plt.close('all')
display(Image(filename=str(anim_path_basin)))
print('Done.')

---
## 8. Summary of output files

In [ ]:
print('=== figures_2d/ ===')
for p in sorted(cfg.figures_2d_dir.glob('*')):
    print(f'  {p.name}')

print()
print('=== results_2d/ ===')
for p in sorted(cfg.results_2d_dir.glob('*')):
    print(f'  {p.name}')

---
## 9. Limitations and next steps

The 2D SH-wave model extends the 1D velocity-stress formulation by adding
lateral stress gradients.  The homogeneous case verifies basic wave speed
and wavefront behavior.  The layered case demonstrates reflection,
reverberation, and free-surface response.  The Gaussian basin shows how
2D geometry produces edge-generated surface waves and lateral amplitude
variability that the 1D model cannot predict.

**Known limitations of this first implementation:**

| Limitation | Consequence | Remedy |
|---|---|---|
| Simple Gaussian damping at boundaries | Reflections from PML-free edges at late times | Replace with Perfectly Matched Layer (PML) |
| Second-order staggered FD only | Numerical dispersion at high f or coarse grids | Use 4th-order spatial operators |
| No material damping (Q) | Waves do not attenuate with propagation | Add memory-variable anelastic term |
| Point source only | No plane-wave or vertically incident input | Add plane-wave / 1D-input boundary |
| Regular Cartesian grid | Cannot represent topography | Use boundary-conforming or unstructured mesh |

**Potential next steps:**
- Grid refinement convergence study for the 2D arrival time
- Quantitative 1D-vs-2D surface motion comparison (centerline check)
- Full SH-to-PSV extension (add $v_x$, $v_z$, $\\sigma_{xx}$, $\\sigma_{zz}$, $\\sigma_{xz}$)
- Engineering amplification functions from 2D surface arrays